## Core Question

Previously, we had a simple plan: measure the value of each action in each state, then always pick the most valuable action!

Sounds great! But how do we actually calculate these values?

Let's make some HUGE simplifications.

### The vocabulary (do not conflate these!)

| Item | Symbol | The simplest question |
|---|---|---|
| **State** | $s$ | What information completely describes the world *for predicting its next step*? |
| **Observation** | $o$ | What can **this agent** actually see? |
| **Action** | $a$ | What can the agent choose? |
| **Policy** | $\pi(a\mid o)$ or $\pi(a\mid s)$ | How does the agent select actions? |
| **Transition dynamics** | $P(s'\mid s,a)$ | What happens to the world after an action? |
| **Reward** | $R(s,a,s')$ | How does the task score the outcome? |
| **Objective** | $\mathbb E[\sum_t\gamma^t R_{t+1}]$ | What are we trying to maximize? |


Assume we have:
- Finite states and actions: We know every possible state and every action we can take.
- Fully observable, Markov states: We can always observe the complete current state, which **contains everything needed to predict what happens next**.
- Known transitions: We know exactly how the environment works, including the probability of every possible next state.
- Known rewards: We know how much reward each possible outcome gives us.
- Stationary environment: The rules of the world never change.

In other words, we know exactly how our little world works.

teaching idea: 
>Start with a world where everything is known. Solve it exactly. Then remove the assumptions, one by one.

So, is that enough?

**In theory, YES!**

But here's the problem: **the value of an action isn't just its immediate reward. It includes all the future rewards that action might lead to.**

The killer word here is **All future reward**.

Let's go over an example to show this.

Imagine we're in state S, and we have two choices:

- **Action A:** Get 1.5 reward immediately, and the game ends.
- **Action B:** Get 1 reward immediately, and return to state S.

Let's set $\gamma=0.5$.

Which action is better?

Action A is easy. Its total reward is 1.5.

But Action B is interesting. We can keep choosing it forever!

If we do, the total discounted reward is:

$$
1+0.5+0.25+0.125+\cdots=2
$$

So Action B is actually better!

This example is very simple and we can calculate by hand.

Now imagine hundreds of states, each with multiple actions, probabilistic transitions, and potentially endless loops.

Try calculate by hand every possible future trajectory, all the way to the end 😂.

The Core question is:
>How on earth can we calculate the value of an action when its consequences extend **indefinitely into the future**?

Bellman framework helps us achieve this:

$$
\underbrace{\text{Entire future}}_{\text{Global problem}}
\quad\longrightarrow\quad
\underbrace{\text{One step}+\text{Future value}}_{\text{Local problem}}
$$

It says:
>I can turn an infinite-horizon optimization problem into a recursive equation, prove that it has a unique solution, and construct an algorithm guaranteed to converge to that solution.

Sounds like magic?

Let's go over how it works!

To understand that, we first need a little mathematics.

### Fixed Point Theory Intuition

Say we have a Equation：

$$
V=1+0.5V
$$

once it reaches 2, it won't change, this is fixed point.

In [2]:
# this experiment proves the point above
v = 0.0

for i in range(7):
    print(f"Iteration {i}: V = {v:.5f}")
    v = 1 + 0.5 * v

Iteration 0: V = 0.00000
Iteration 1: V = 1.00000
Iteration 2: V = 1.50000
Iteration 3: V = 1.75000
Iteration 4: V = 1.87500
Iteration 5: V = 1.93750
Iteration 6: V = 1.96875


### Why it will converge?

Assume fixed point is $x^*=2 $, our guess is x.

Apply it once:

\begin{aligned}
|f(x)-f(x^*)|
&=|(1+0.5x)-(1+0.5x^*)|\\
&=0.5|x-x^*|
\end{aligned}

See? we are 50% closer! This is the intuition for **contraction**. More complicated Math proof will look like this, which we will skip:)

When satisfying certain contraction mapping，Banach Fixed Point Theorem say:
- exists a fixed point
- any starting point will converge to it



### Bellman operator and equation

Let's review some ideas.

Return is defined as: 

$$
G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots
$$

and we can derive this based on above:

$$
G_t=R_{t+1}+\gamma G_{t+1}
$$

and then after some not very complicated math (which we will skip 😂), we get:

Bellman Expectation Equation:

$$
V^\pi(s)=\mathbb E_\pi[R+\gamma V^\pi(S')]
$$

Basically it describes the math relationship for value function.

Now comes the magic, if we turn this into a operator (a.k.a, function in, function out). It's called Bellman Operator:

$$
(T^\pi V)(s)=\mathbb E_\pi[r+\gamma V(s')\mid s]
$$

and repeatedly applying this operator to any initial function converges to a fixed point (a function now)!

---
### What is the use of it?

What we want is: for a given policy and reward and gamma etc
>What is the state/action's value function under such setup?
>
>Note: state and value each has a corresponding Bellman equation, same idea though.

All we care is whether/how we can know the value of everything (a.k.a value equation)!

After some magic math involving Bellman Equation/Operator + fixed point theory, the conclusion is:
>Find a random starting point, keep iterating, and it will converge to a fixed value function.
>
>hooyah! it tell us random starting point + iterating can lead us to what we want.

---
### Is it enough?

Actually there is an efficient issue.

RL's goal is to find a policy, which could maximize expected return at each state:

$$
V^*(s)=\max_\pi V^\pi(s)
$$

The Bellman Expectation Equation is good, to find the optimal policy, 1 way is to iterate through all policies, compare everyone's value function, then pick the best.

Sounds tedious!

Now let's do a little modification to the existing Bellman operator:

$$
(\mathcal T^\pi V)(s)=
\mathbb E_{a\sim\pi}[R+\gamma V(S')]
$$

and change it to (called **Bellman Optimality Operator**):

$$
(\mathcal T^*V)(s)=
\max_a\mathbb E[R+\gamma V(S')\mid s,a]
$$

The difference is:
>Bellman operator: **policy-weighted average** of expected returns
>
>Bellman optimal operator: **maximum** expected return.

<br>

Note, there are 2 sources of randomness in RL:
>under a policy, which action an agent picks **could be** probabilistic
>
>**KEY:** Given an action, the reward and next state **could be** probabilistic
>

the expected return for a action could take into environment's randomness factor:
>Ex. 40% +6, 60% +10, then expected return for this action is 0.4 * 6 + 0.6 * 10 = 8.4

policy-weighted average is:
>after we calculated the expected return of each action, then on top of it, doing policy-weighted average.


---
The Bellman Optimality Operator is a contraction, so it has a unique fixed point. We can further show that this fixed point is exactly the optimal value function $V^*$, and a greedy policy with respect to $V^*$ is optimal.

>Contraction explains why iteration converges. Bellman optimality explains why the destination is what we want.

Proof is skipped.

This means:
>Just by changing how we pick action. Just by following Bellman optimal operator and iterating, we can find the optimal value function, build a policy using it, and achieve our goal. No need to compare all possible polices!